# 06 · Train a neural network (plan steps 3.3, 4.1–4.2)

One notebook for every neural model: set `CONFIG` and run all. For stage 3 it is `configs/cnn.yaml`
(CNN from scratch, D-012). Select a **T4 GPU** runtime.

What happens (`plantdx.train`): the `tf.data` pipeline from `configs/base.yaml` (train cached on local disk,
augmentation on the GPU as the first model layers) → training with checkpoint on **val macro-F1** and early
stopping / learning-rate decay on **val loss** → the best epoch is scored on val → one row in the results table.

Everything for the run is written to Drive (`artifacts/runs/<run_id>/`), so a session reset loses at most
the epoch in progress. The first epoch is slower: it decodes all JPEGs into the disk cache.

Expected time for `cnn.yaml`: roughly 1–1.5 min per epoch on a T4, up to 30 epochs (early stopping usually
ends sooner).

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (D-009)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L || echo "no GPU"
!python -m plantdx.env

## 2 · Images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

## 3 · Train

In [ ]:
CONFIG = "configs/cnn.yaml"

from plantdx.train import train

report = train(CONFIG, DATA_ROOT)
RUN_ID = report["run_id"]
report["metrics"]

## 4 · Learning curves ← **look at this figure**

Dotted line: the epoch kept (best val macro-F1). Check that val loss does not climb steadily while train loss
falls (overfitting), and that the curves have flattened rather than being cut off early.

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, display

from plantdx.paths import run_dir
from plantdx.viz import plot_learning_curves

FIG = Path("/content/figures"); FIG.mkdir(exist_ok=True)
curves = plot_learning_curves(run_dir(RUN_ID) / "history.csv", FIG / f"curves_{RUN_ID}.png", title=RUN_ID)
display(IPImage(str(curves)))

## 5 · Weakest classes on val

In [ ]:
import pandas as pd

per_class = pd.DataFrame(report["per_class"]).sort_values("f1")
per_class.head(8)[["class", "support", "precision", "recall", "f1"]]

## 6 · Download the results

Unzip at the **repository root**: `reports/results.csv`, `reports/runs/<run_id>/` (metrics, history, config,
environment) and `reports/figures/curves_<run_id>.png`. The model (`best.keras`) stays on Drive.

In [ ]:
from google.colab import files
from plantdx.results import export_for_repo

out = export_for_repo([RUN_ID], f"/content/{RUN_ID}.zip", figures=[curves])
files.download(str(out))